In [0]:
import os
import requests
from dotenv import load_dotenv
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
BRONZE_NAME_TABLE = "sandbox_prd.bronze_layer.movies_and_series"
BRONZE_NAME_TABLE_API = "sandbox_prd.bronze_layer.movies_and_series_api"
OMDB_KEY = os.getenv('OMDB_KEY')
BASE_URL = f'http://www.omdbapi.com/'
results = []

In [0]:
load_dotenv()

In [0]:
#read table
df = spark.read.table(BRONZE_NAME_TABLE)
rows = df.select('name_en').collect()

for show in rows:
  params = {
    'apikey': OMDB_KEY,
    't': show['name_en']
  }
  req = requests.get(BASE_URL, params=params)
  data = req.json()
  results.append({
    'title_1': show['name_en'],
    'title': data.get('Title'),
    'year': data.get('Year'),
    'rated': data.get('Rated'),
    'released': data.get('Released'),
    'runtime': data.get('Runtime'),
    'genre': data.get('Genre'),
    'country': data.get('Country'),
    'poster': data.get('Poster'),
    'metascore': data.get('Metascore'),
    'imdbRating': data.get('imdbRating'),
    'imdbVotes': data.get('imdbVotes'),
    'totalSeasons': data.get('totalSeasons')
  })

In [0]:
schema = StructType([
    StructField('title_1', StringType(), True),
    StructField('title', StringType(), True),
    StructField('year', StringType(), True),
    StructField('rated', StringType(), True),
    StructField('released', StringType(), True),
    StructField('runtime', StringType(), True),
    StructField('genre', StringType(), True),
    StructField('country', StringType(), True),
    StructField('poster', StringType(), True),
    StructField('metascore', StringType(), True),
    StructField('imdbRating', StringType(), True),
    StructField('imdbVotes', StringType(), True),
    StructField('totalSeasons', StringType(), True)
])

In [0]:
# Create dataframe
df_api = spark.createDataFrame(results,schema=schema)
df_api.write.format("delta").mode("overwrite").saveAsTable(BRONZE_NAME_TABLE_API)

In [0]:
%sql
select
*
from sandbox_prd.bronze_layer.movies_and_series_api